# Shape-only systematic uncertainties

Reproduce Figure 7 and Section 5.1.

In [ ]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/hnsbi_asimov")
    repo = project / "repository"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/rafaellopesdesa/hnsbi_asimov.git", str(repo)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")],
        check=True,
    )
else:
    repo = Path.cwd()
    project = repo
sys.path.insert(0, str(repo))
from utils import setup_workspace

setup_workspace(repo, project / "workspace")

## Generate $p_s(\mathbf{x};\alpha_{\text{scale}}=\pm1)$

In [ ]:
if not Path("dataframes/signal_scale_down.parquet").exists():
    subprocess.run(
        [sys.executable, str(repo / "generate_distributions.py"), "--systematics-only"], check=True
    )

In [ ]:
import gc
from pathlib import Path
import jax

jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from iminuit import Minuit
from utils import (
    FEATURES,
    density_ratio_trainer,
    predict_with_model,
    as_inference_session,
    evaluate_ratio_packs,
    load_ratio_pack,
    ratio_training_dataframe,
    sample_selected_flow,
)
from utils_nf import collect_preselected_parquet, load_flow
from utils_systematics import ReferenceNormalizedSystematicsModel

FEATURES = list(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = Path("dataframes")
SAMPLE_PATHS = {
    ("signal", "nominal"): BASE_PATH / "signal.parquet",
    ("signal", "up"): BASE_PATH / "signal_scale_up.parquet",
    ("signal", "down"): BASE_PATH / "signal_scale_down.parquet",
    ("background", "nominal"): BASE_PATH / "background.parquet",
    ("background", "up"): BASE_PATH / "background_scale_up.parquet",
    ("background", "down"): BASE_PATH / "background_scale_down.parquet",
}
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
REFERENCE_FLOW_TYPE = "quadratic_spline"
NOMINAL_RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
SYSTEMATIC_RATIO_MODEL_DIR = {
    (process, direction): Path(f"models_systematics_{process}_scale_{direction}_vs_nominal")
    for process in ["signal", "background"]
    for direction in ["up", "down"]
}
SYSTEMATIC_OUTPUT_DIR = Path("saved_densities_systematics")
for directory in [*SYSTEMATIC_RATIO_MODEL_DIR.values(), SYSTEMATIC_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.92
STREAM_BATCH_SIZE = 100000
MAX_SYSTEMATIC_TRAIN_EVENTS = 1000000
MAX_EVAL_EVENTS = 250000
RATIO_HIDDEN_LAYERS = 4
RATIO_NEURONS = 1024
RATIO_N_EPOCHS = 50
RATIO_BATCH_SIZE = 4096
RATIO_LEARNING_RATE = 0.001
RATIO_HOLDOUT_FRACTION = 0.25
RATIO_VALIDATION_FRACTION = 0.2
RATIO_PATIENCE = 10
RATIO_LOAD_IF_AVAILABLE = True
RATIO_EVALUATION_BATCH_SIZE = 100000
ASIMOV_MU_TRUE = 1.0
ASIMOV_REFERENCE_EVENTS = 1000000
REFERENCE_SAMPLING_BATCH_SIZE = 65536

## Load $q_{\boldsymbol{\phi}}$, $r_{s,\boldsymbol{\psi}}$ and the fixed preselection

In [ ]:
PRESEL_pack = load_ratio_pack(PRESEL_MODEL_DIR, 0)
NOMINAL_RATIO_MODELS = {
    s: [load_ratio_pack(path, i) for i in range(4)] for s, path in NOMINAL_RATIO_MODEL_DIR.items()
}


def evaluate_PRESEL_ratio(x):
    return predict_with_model(x, **PRESEL_pack)


state = np.load(PRESEL_MODEL_DIR / "selection.npz")
PRESEL_RATIO_CUT = float(state["ratio_cut"])

## Prepare the selected samples with $\lambda_s(\alpha_{\text{scale}})=\lambda_s(0)$

In [ ]:
SELECTED_SAMPLES = {}
SELECTED_YIELD = {}
for sample_index, ((process, variation), path) in enumerate(SAMPLE_PATHS.items()):
    samples, stats = collect_preselected_parquet(
        path,
        features=FEATURES,
        ratio_predictor=evaluate_PRESEL_ratio,
        ratio_cut=PRESEL_RATIO_CUT,
        max_train_events=MAX_SYSTEMATIC_TRAIN_EVENTS,
        max_eval_events=MAX_EVAL_EVENTS,
        batch_size=STREAM_BATCH_SIZE,
        presel_fraction=PRESEL_TRAIN_FRACTION,
        flow_train_fraction=FLOW_TRAIN_FRACTION,
        split_seed=SPLIT_SEED,
        reservoir_seed=SEED + 1000 + 100 * sample_index,
    )
    selected_yield = float(state[f"lambda_{process}"])
    SELECTED_YIELD[process, variation] = float(selected_yield)
    for split, dataframe in samples.items():
        retained_weight = float(dataframe["weight"].sum())
        dataframe["weight"] *= selected_yield / retained_weight
    SELECTED_SAMPLES[process, variation] = samples

## Train $g_{s,\eta}^{\pm}(\mathbf{x})$ (Section 5.1)

In [ ]:
def train_systematic_ratio(process, direction, seed):
    key = (process, direction)
    varied_train = SELECTED_SAMPLES[key]["flow_train"]
    nominal_train = SELECTED_SAMPLES[process, "nominal"]["flow_train"]
    training_dataframe = ratio_training_dataframe(
        varied_train, nominal_train, MAX_SYSTEMATIC_TRAIN_EVENTS, seed
    )
    trainer = density_ratio_trainer(
        dataset=training_dataframe,
        weights=training_dataframe["weights_normed"].to_numpy(),
        training_labels=training_dataframe["train_labels"].to_numpy(),
        features=FEATURES,
        features_scaling=FEATURES,
        sample_name=[f"{process} scale {direction}", f"{process} nominal"],
        output_name=f"{process}_scale_{direction}",
        path_to_figures=f"{SYSTEMATIC_RATIO_MODEL_DIR[key]}/",
        path_to_models=f"{SYSTEMATIC_RATIO_MODEL_DIR[key]}/",
    )
    trainer.train(
        hidden_layers=RATIO_HIDDEN_LAYERS,
        neurons=RATIO_NEURONS,
        number_of_epochs=RATIO_N_EPOCHS,
        batch_size=RATIO_BATCH_SIZE,
        learning_rate=RATIO_LEARNING_RATE,
        scalerType="MinMax",
        ensemble_index=None,
        verbose=1,
        rnd_seed=seed,
        holdout_split=RATIO_HOLDOUT_FRACTION,
        validation_split=RATIO_VALIDATION_FRACTION,
        callback_patience=RATIO_PATIENCE,
        num_workers=0,
        load_trained_models=RATIO_LOAD_IF_AVAILABLE,
        calibration=False,
    )
    pack = {"scaler": trainer.scaler, "model": as_inference_session(trainer.model_NN)}
    nominal_eval = SELECTED_SAMPLES[process, "nominal"]["eval"]
    raw_nominal_ratio = evaluate_ratio_packs(
        [pack], nominal_eval[FEATURES], batch_size=RATIO_EVALUATION_BATCH_SIZE
    )
    normalization = float(
        np.average(raw_nominal_ratio, weights=nominal_eval["weight"].to_numpy(dtype=np.float64))
    )
    del (trainer, training_dataframe, raw_nominal_ratio)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (pack, normalization)


SYSTEMATIC_RATIO_MODELS = {}
SYSTEMATIC_RATIO_NORMALIZATION = {}
for process_index, process in enumerate(["signal", "background"]):
    for direction_index, direction in enumerate(["up", "down"]):
        pack, normalization = train_systematic_ratio(
            process, direction, SEED + 20000 + 1000 * process_index + 100 * direction_index
        )
        SYSTEMATIC_RATIO_MODELS[process, direction] = pack
        SYSTEMATIC_RATIO_NORMALIZATION[process, direction] = normalization

## Construct $\mathcal A_M(\boldsymbol{\theta}_A)$ at $(\mu_A,\alpha_A)=(1,0)$

In [ ]:
reference_flow = load_flow(
    "reference", model_dir=REFERENCE_FLOW_MODEL_DIR, flow_type=REFERENCE_FLOW_TYPE, device=device
)
np.random.seed(SEED + 30000)
torch.manual_seed(SEED + 30000)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED + 30000)
reference_values, reference_flow_acceptance = sample_selected_flow(
    reference_flow,
    ASIMOV_REFERENCE_EVENTS,
    lambda x: evaluate_PRESEL_ratio(x) >= PRESEL_RATIO_CUT,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
asimov_dataset = pd.DataFrame(reference_values, columns=FEATURES)
del reference_values, reference_flow
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
NOMINAL_RATIOS_ASIMOV = {}
for process in ["signal", "background"]:
    raw_reference_ratio = evaluate_ratio_packs(
        NOMINAL_RATIO_MODELS[process],
        asimov_dataset[FEATURES],
        batch_size=RATIO_EVALUATION_BATCH_SIZE,
    )
    normalization = float(raw_reference_ratio.mean())
    normalized_ratio = raw_reference_ratio / normalization
    NOMINAL_RATIOS_ASIMOV[process] = normalized_ratio
SYSTEMATIC_RATIOS_ASIMOV = {}
for process in ["signal", "background"]:
    for direction in ["up", "down"]:
        key = (process, direction)
        normalized_systematic_ratio = (
            evaluate_ratio_packs(
                [SYSTEMATIC_RATIO_MODELS[key]],
                asimov_dataset[FEATURES],
                batch_size=RATIO_EVALUATION_BATCH_SIZE,
            )
            / SYSTEMATIC_RATIO_NORMALIZATION[key]
        )
        SYSTEMATIC_RATIOS_ASIMOV[key] = normalized_systematic_ratio
nominal_signal_yield = SELECTED_YIELD["signal", "nominal"]
nominal_background_yield = SELECTED_YIELD["background", "nominal"]
asimov_weights = (
    ASIMOV_MU_TRUE * nominal_signal_yield * NOMINAL_RATIOS_ASIMOV["signal"]
    + nominal_background_yield * NOMINAL_RATIOS_ASIMOV["background"]
) / len(asimov_dataset)
ARRAY_PATHS = {
    "weights": SYSTEMATIC_OUTPUT_DIR / "weights_asimov.npy",
    ("signal", "nominal"): SYSTEMATIC_OUTPUT_DIR / "ratio_signal_nominal.npy",
    ("background", "nominal"): SYSTEMATIC_OUTPUT_DIR / "ratio_background_nominal.npy",
}
for process in ["signal", "background"]:
    for direction in ["up", "down"]:
        ARRAY_PATHS[process, direction] = (
            SYSTEMATIC_OUTPUT_DIR / f"ratio_{process}_scale_{direction}_over_nominal.npy"
        )
np.save(ARRAY_PATHS["weights"], asimov_weights)
for process in ["signal", "background"]:
    np.save(ARRAY_PATHS[process, "nominal"], NOMINAL_RATIOS_ASIMOV[process])
    for direction in ["up", "down"]:
        np.save(ARRAY_PATHS[process, direction], SYSTEMATIC_RATIOS_ASIMOV[process, direction])

## Normalize $\widetilde g_{s,\eta}(\mathbf{x};\alpha)$ on $\mathcal X_M$ (Section 5.1)

In [ ]:
def make_systematics_workspace():
    samples = []
    for process in ["signal", "background"]:
        nominal_yield = SELECTED_YIELD[process, "nominal"]
        modifiers = []
        if process == "signal":
            modifiers.append({"name": "mu", "type": "normfactor", "data": None})
        modifiers.append(
            {
                "name": "scale",
                "type": "normplusshape",
                "data": {
                    "hi_data": [1.0],
                    "lo_data": [1.0],
                    "hi_ratio": str(ARRAY_PATHS[process, "up"]),
                    "lo_ratio": str(ARRAY_PATHS[process, "down"]),
                },
            }
        )
        samples.append(
            {
                "name": process,
                "data": [nominal_yield],
                "ratios": str(ARRAY_PATHS[process, "nominal"]),
                "modifiers": modifiers,
            }
        )
    return {
        "channels": [
            {
                "name": "SR",
                "type": "unbinned",
                "weights": str(ARRAY_PATHS["weights"]),
                "samples": samples,
            }
        ],
        "measurements": [
            {
                "name": "meas",
                "config": {
                    "poi": "mu",
                    "parameters": [
                        {"name": "mu", "inits": [1.0], "bounds": [[0.0, 3.0]]},
                        {"name": "scale", "inits": [0.0], "bounds": [[-5.0, 5.0]]},
                    ],
                },
            }
        ],
        "version": "1.0.0",
    }


ws_systematics = make_systematics_workspace()

## Figure 7(a): likelihood in $(\mu,\alpha_{\text{scale}})$

In [ ]:
model_systematics = ReferenceNormalizedSystematicsModel(
    workspace=ws_systematics, measurement_to_fit="meas"
)
list_parameters, initial_values = model_systematics.get_model_parameters()
initial_values = np.asarray(initial_values, dtype=float)
PARAMETER_BOUNDS = {"mu": (0, 3), "scale": (-5, 5)}


def make_bounded_minuit(start_values=None, fit_strategy=0):
    values = initial_values if start_values is None else np.asarray(start_values, dtype=float)
    minimizer = Minuit(
        model_systematics.model,
        values,
        grad=model_systematics.model_grad,
        name=tuple(list_parameters),
    )
    minimizer.errordef = Minuit.LEAST_SQUARES
    minimizer.strategy = int(fit_strategy)
    for parameter, bounds in PARAMETER_BOUNDS.items():
        if parameter in list_parameters:
            minimizer.limits[parameter] = bounds
    return minimizer


def perform_bounded_profile_scan(
    parameter_name, bound_range, freeze_params=(), fit_strategy=0, size=50
):
    minimizer = make_bounded_minuit(fit_strategy=fit_strategy)
    for parameter in freeze_params:
        minimizer.fixed[parameter] = True
    scan_points, nll_values, _ = minimizer.mnprofile(
        parameter_name, bound=bound_range, subtract_min=True, size=int(size)
    )
    return (scan_points, nll_values)


minuit_global = make_bounded_minuit()
minuit_global.migrad()
best_fit = dict(zip(list_parameters, np.asarray(minuit_global.values)))
MU_LIKELIHOOD_RANGE = (0.0, 2.0)
ALPHA_LIKELIHOOD_RANGE = (-0.03, 0.03)
MU_LIKELIHOOD_POINTS = 51
ALPHA_LIKELIHOOD_POINTS = 61
LIKELIHOOD_EVALUATION_BATCH_SIZE = 4
mu_likelihood_values = np.linspace(*MU_LIKELIHOOD_RANGE, MU_LIKELIHOOD_POINTS)
alpha_likelihood_values = np.linspace(*ALPHA_LIKELIHOOD_RANGE, ALPHA_LIKELIHOOD_POINTS)
mu_mesh, alpha_mesh = np.meshgrid(mu_likelihood_values, alpha_likelihood_values, indexing="xy")
likelihood_points = np.column_stack([mu_mesh.ravel(), alpha_mesh.ravel()])
nll_minimum = float(
    model_systematics.model(np.array([best_fit["mu"], best_fit["scale"]], dtype=float))
)
delta_nll_2d = model_systematics.model_many(
    likelihood_points, batch_size=LIKELIHOOD_EVALUATION_BATCH_SIZE
).reshape(alpha_mesh.shape)
delta_nll_2d = np.maximum(delta_nll_2d - nll_minimum, 0.0)
fig, ax = plt.subplots(figsize=(8.2, 6.2))
color_mesh = ax.pcolormesh(
    mu_mesh, alpha_mesh, np.minimum(delta_nll_2d, 15.0), shading="auto", cmap="viridis"
)
contour_levels = [2.3, 5.99, 9.21]
contours = ax.contour(
    mu_mesh, alpha_mesh, delta_nll_2d, levels=contour_levels, colors="white", linewidths=1.4
)
ax.clabel(contours, fmt={2.3: "68%", 5.99: "95%", 9.21: "99%"}, inline=True, fontsize=9)
ax.scatter(
    [ASIMOV_MU_TRUE],
    [0.0],
    marker="x",
    s=90,
    linewidths=2.0,
    color="white",
    label="Asimov point",
    zorder=5,
)
ax.scatter(
    [best_fit["mu"]],
    [best_fit["scale"]],
    marker="*",
    s=120,
    color="C3",
    label="profiled best fit",
    zorder=6,
)
fig.colorbar(color_mesh, ax=ax, label="$\\Delta(-2\\log L)$")
ax.set_xlabel("$\\mu$")
ax.set_ylabel("$\\alpha_{\\mathrm{scale}}$")
ax.set_title("two-dimensional Asimov likelihood")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## Figure 7(b): $t_\mu$ with $\alpha_{\text{scale}}$ fixed or profiled

In [ ]:
MU_SCAN_RANGE = (0.0, 3.0)
MU_SCAN_POINTS = 61
scan_profiled, tmu_profiled = perform_bounded_profile_scan(
    parameter_name="mu",
    freeze_params=[],
    bound_range=MU_SCAN_RANGE,
    fit_strategy=0,
    size=MU_SCAN_POINTS,
)
scan_fixed, tmu_fixed = perform_bounded_profile_scan(
    parameter_name="mu",
    freeze_params=["scale"],
    bound_range=MU_SCAN_RANGE,
    fit_strategy=0,
    size=MU_SCAN_POINTS,
)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(scan_fixed, tmu_fixed, lw=2, ls="--", label="scale fixed: $\\alpha_{\\mathrm{scale}}=0$")
ax.plot(scan_profiled, tmu_profiled, lw=2, label="scale profiled")
ax.axvline(1.0, color="black", ls=":", lw=1, alpha=0.7)
for level in [1.0, 4.0]:
    ax.axhline(level, color="0.75", ls=":", lw=1)
ax.set_xlim(*MU_SCAN_RANGE)
ax.set_ylim(bottom=0.0)
ax.set_xlabel("$\\mu$")
ax.set_ylabel("$t_\\mu$")
ax.legend()
ax.set_title("signal-strength closure")
fig.tight_layout()
plt.show()